# Spring A-density and foreground grammar v0.1

Locked F2-C / 70% existing + 30% selective foreground. The experiment changes only A/C path grammar. B/D and existing vegetation vectors remain identical. No object generator, final color system or G-code runs here.

See `docs/experiments/spring_density_rhythm_v01/README.md` and `geumgang_source_audit.md`. The historical V39.10 file is read only; its final-production identity is provisional.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Image, Markdown
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "src/datascape").is_dir())
sys.path.insert(0, str(root / "src"))
master = Path(os.environ.get("DATASCAPE_MASTER", root / "data/raw/SeoulNorth_v02_MASTER_30m.tif"))
veg = Path(os.environ.get("DATASCAPE_VEG", root / "data/raw/SeoulNorth_v02_VEG_10m.tif"))
output = Path(os.environ.get("DATASCAPE_OUTPUT", root / "docs/experiments/spring_density_rhythm_v01"))
config_path = root / "configs/scenes/seoul_spring_density_rhythm_v01.json"
assert master.is_file() and veg.is_file(), "Set DATASCAPE_MASTER and DATASCAPE_VEG to the original TIFFs."
config = json.loads(config_path.read_text())
print(config["study"], "seed", config["seed"])

The preserved `0317test` code varies its row step with squared row-maximum DEM height. `density_wave` changes pressure and color-path eligibility; its row-step contribution is a constant mean. Historical depth warps and horizon resets are deliberately not transferred.

In [ ]:
historical = root / "artworks/2026-03-17_geumgang/original/source.py"
lines = historical.read_text().splitlines()
print("\n".join(f"{i+1}: {lines[i]}" for i in range(220, 239)))

In [ ]:
from datascape.rendering.density_rhythm_study import run
record = run(master, veg, config_path, output, root)

In [ ]:
rows = ["| Variant | Panel paths | Total length (m) | A (m) | C (m) |", "| --- | ---: | ---: | ---: | ---: |"]
for name, item in record["metrics"].items():
    count = sum(v["vectors"] for v in item["layers"].values())
    rows.append(f"| {name} | {count} | {item['all_length_mm']/1000:.2f} | {item['layers']['A']['length_mm']/1000:.2f} | {item['layers']['C']['length_mm']/1000:.2f} |")
display(Markdown("\n".join(rows)))
display(Image(filename=str(output / "grammar_comparison.png")))

In [ ]:
display(Image(filename=str(output / "A_density_isolation.png")))

In [ ]:
display(Image(filename=str(output / "foreground_comparison.png")))
display(Image(filename=str(output / "foreground_black_comparison.png")))
display(Image(filename=str(output / "foreground_layers_comparison.png")))

In [ ]:
display(Image(filename=str(output / "production_density_comparison.png")))
display(Image(filename=str(output / "preview_modes.png")))

In [ ]:
display(Image(filename=str(output / "density_mist_diagnostic.png")))
display(Image(filename=str(output / "source_spacing_profile.png")))
display(Image(filename=str(output / "mist_vector_comparison.png")))

In [ ]:
display(Image(filename=str(output / "physical_pair_comparison.png")))
display(Image(filename=str(output / "blended_color_ready_physical.png")))

In [ ]:
from datascape.strokes.vector import read_vectors
from datascape.strokes.density_rhythm import layer_digest, assign_material
baseline = read_vectors(output / "baseline.json.gz")["strokes"]
selected = read_vectors(output / "blended_color_ready.json.gz")["strokes"]
for layer in ["B", "D", "V"]:
    assert layer_digest(baseline, layer) == layer_digest(selected, layer)
c = [s for s in selected if s["layer"] == "C"]
color = assign_material(c, "color_support")
assert [s["xy_mm"] for s in c] == [s["xy_mm"] for s in color]
assert [s["width_mm"] for s in c] == [s["width_mm"] for s in color]
print(json.dumps(record["invariants"], indent=2))
print(json.dumps(record["local_density"]["blended_color_ready"], indent=2))

Opacity is a display simulation. It does not establish physical ink dilution, paper absorption, brush pressure or a hardware Z coordinate. The canonical saved centerlines are shared by previews and future machine export. `C_material_options.json` records intent options without duplicating the geometry or choosing a final palette color.